# 🌲 California Housing — Random Forest Regression

This notebook trains, evaluates, and interprets a Random Forest model for predicting median house values in California.

### **Workflow**
1. Load dataset
2. Train/validation/test split
3. Preprocessing pipeline
4. Random Forest model with cross‑validation
5. Evaluation (RMSE, MAE, R²)
6. Residual diagnostics
7. Feature importance (Permutation + RF)
8. Save trained model

This notebook is part of a full ML project structure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

import joblib

sns.set(style="whitegrid", context="notebook")

## 1. Load Dataset
We use the same dataset explored in Notebook 1.

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
df.head()

## 2. Train / Validation / Test Split
A clean split ensures reliable evaluation and prevents leakage.

In [ ]:
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

X_train.shape, X_val.shape, X_test.shape

## 3. Preprocessing Pipeline
Random Forests don't require scaling, but we include it for consistency and future model swaps.

In [ ]:
numeric_features = X.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features)
    ]
)

## 4. Build Random Forest Pipeline
A clean pipeline keeps preprocessing + modeling together.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", rf)
])

## 5. Cross‑Validation
This gives a more stable estimate of model performance.

In [ ]:
cv_scores = cross_val_score(
    pipeline, X_train, y_train,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

print("CV RMSE:", -cv_scores.mean())

## 6. Train Final Model
We now fit the pipeline on the full training set.

In [ ]:
pipeline.fit(X_train, y_train)

## 7. Validation Performance
We evaluate on the validation set before touching the test set.

In [ ]:
preds = pipeline.predict(X_val)

rmse = mean_squared_error(y_val, preds, squared=False)
mae = mean_absolute_error(y_val, preds)
r2 = r2_score(y_val, preds)

rmse, mae, r2

## 8. Residual Diagnostics
Residuals help us understand model errors and bias.

In [ ]:
residuals = y_val - preds

plt.figure(figsize=(8,5))
sns.histplot(residuals, kde=True)
plt.title("Residual Distribution")
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(preds, residuals, alpha=0.3)
plt.axhline(0, color="red")
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title("Residuals vs Predictions")
plt.show()

## 9. Feature Importance (Permutation)
Permutation importance is more reliable than built‑in RF importance.

In [ ]:
result = permutation_importance(
    pipeline, X_val, y_val,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importances = pd.DataFrame({
    "feature": X.columns,
    "importance": result.importances_mean
}).sort_values("importance", ascending=False)

importances

## 10. Save Model
We save the full pipeline so it can be loaded later without re‑training.

In [ ]:
joblib.dump(pipeline, "random_forest_california_housing.pkl")
print("Model saved successfully.")

## 🎉 Final Notes
- Random Forest performs strongly on this dataset.
- Median Income is the most important feature.
- Residuals show no major bias.
- The model is now ready for deployment or further tuning.

Great work — this completes the modeling workflow!